# Lab D — Your First Kernels

**GPU Mastery · CUDA** · run on the **2× RTX 5090** server (WSL2)

You've seen the ideas — a kernel is one function run by a million threads, and *tiling* keeps data on-chip. Now you **write and run** them, and **measure** tiling turning a slow matmul into a fast one.

You will:
1. Write and run a **vector-add** kernel — the 5-step host↔device flow, in real code.
2. Use a **grid-stride loop** to sweep data bigger than your thread count.
3. Write a **naïve matmul**, then a **tiled matmul**, and time the speedup yourself.

> This lab is **raw CUDA C++** compiled with `nvcc`. We write each kernel to a `.cu` file, compile it, and run it — all from notebook cells.

## 0. Requirements

You need the **CUDA Toolkit** (`nvcc`) that supports Blackwell — **CUDA 12.8+**. We compile with `-arch=native`, which auto-targets whatever GPU is present. If your toolkit is older and `native` fails, swap it for `-arch=sm_120`.

In [ ]:
!nvcc --version
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

## 1. Vector add — the five steps, in real code

The kernel does one add per thread. `main()` walks the exact 5-step flow from the lecture: **allocate on GPU → copy in → launch → copy out → free**.

In [ ]:
%%writefile vadd.cu
#include <cstdio>
#include <cstdlib>

__global__ void add(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // this thread's slot
    if (i < n) c[i] = a[i] + b[i];                    // one add
}

int main() {
    int n = 1 << 20;                       // ~1 million elements
    size_t bytes = n * sizeof(float);

    float *ha = (float*)malloc(bytes), *hb = (float*)malloc(bytes), *hc = (float*)malloc(bytes);
    for (int i = 0; i < n; i++) { ha[i] = 1.0f; hb[i] = 2.0f; }

    float *da, *db, *dc;
    cudaMalloc(&da, bytes); cudaMalloc(&db, bytes); cudaMalloc(&dc, bytes);   // 1. allocate on GPU
    cudaMemcpy(da, ha, bytes, cudaMemcpyHostToDevice);                        // 2. copy inputs in
    cudaMemcpy(db, hb, bytes, cudaMemcpyHostToDevice);

    int threads = 256, blocks = (n + threads - 1) / threads;
    add<<<blocks, threads>>>(da, db, dc, n);                                 // 3. launch

    cudaMemcpy(hc, dc, bytes, cudaMemcpyDeviceToHost);                        // 4. copy result out

    printf("c[0] = %.1f   c[%d] = %.1f   (expected 3.0)\n", hc[0], n - 1, hc[n - 1]);

    cudaFree(da); cudaFree(db); cudaFree(dc);                                 // 5. free
    free(ha); free(hb); free(hc);
    return 0;
}

In [ ]:
!nvcc -arch=native -O2 vadd.cu -o vadd && ./vadd

**Expect:** `c[0] = 3.0   c[1048575] = 3.0`. You just ran a million threads — one add each, all at once.

## 2. Grid-stride loop — more data than threads

You can't always launch one thread per element (data could be billions long). A **grid-stride loop** lets a fixed pool of threads sweep any size: each thread jumps forward by the total thread count and repeats.

In [ ]:
%%writefile vadd_stride.cu
#include <cstdio>

__global__ void add_stride(const float* a, const float* b, float* c, int n) {
    int idx    = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;          // total number of threads
    for (int i = idx; i < n; i += stride)         // sweep the rest
        c[i] = a[i] + b[i];
}

int main() {
    int n = 1 << 24;                              // 16 million elements
    size_t bytes = n * sizeof(float);
    float *a, *b, *c;
    cudaMallocManaged(&a, bytes); cudaMallocManaged(&b, bytes); cudaMallocManaged(&c, bytes);
    for (int i = 0; i < n; i++) { a[i] = 1.0f; b[i] = 2.0f; }

    int threads = 256, blocks = 256;              // only 65,536 threads for 16M elements
    add_stride<<<blocks, threads>>>(a, b, c, n);
    cudaDeviceSynchronize();

    printf("c[0]=%.1f  c[%d]=%.1f  (each thread handled ~%d elements)\n",
           c[0], n - 1, c[n - 1], n / (threads * blocks));
    cudaFree(a); cudaFree(b); cudaFree(c);
    return 0;
}

In [ ]:
!nvcc -arch=native -O2 vadd_stride.cu -o vadd_stride && ./vadd_stride

**Expect:** correct results with each thread handling ~256 elements — the same kernel now scales to any size.

## 3. The payoff: naïve vs tiled matmul

Both kernels compute the same 1024×1024 matmul. The **naïve** one sends every thread to main memory for a full row and column. The **tiled** one loads small tiles into **shared memory** once and reuses them. We time both with CUDA events and print the speedup.

In [ ]:
%%writefile matmul.cu
#include <cstdio>
#include <cstdlib>

#define N    1024      // square matrices, N x N
#define TILE 16        // tile / block size

// --- naïve: every thread reads a full row + column from main memory ---
__global__ void matmul_naive(const float* A, const float* B, float* C, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n && col < n) {
        float sum = 0.0f;
        for (int k = 0; k < n; k++)
            sum += A[row * n + k] * B[k * n + col];
        C[row * n + col] = sum;
    }
}

// --- tiled: load a tile into shared memory once, reuse it many times ---
__global__ void matmul_tiled(const float* A, const float* B, float* C, int n) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];
    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float sum = 0.0f;
    for (int t = 0; t < n / TILE; t++) {
        As[threadIdx.y][threadIdx.x] = A[row * n + t * TILE + threadIdx.x];
        Bs[threadIdx.y][threadIdx.x] = B[(t * TILE + threadIdx.y) * n + col];
        __syncthreads();                                   // wait: tile fully loaded
        for (int k = 0; k < TILE; k++)
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        __syncthreads();                                   // wait: before overwriting
    }
    if (row < n && col < n) C[row * n + col] = sum;
}

int main() {
    int n = N; size_t bytes = n * n * sizeof(float);
    float *hA = (float*)malloc(bytes), *hB = (float*)malloc(bytes), *hC = (float*)malloc(bytes);
    for (int i = 0; i < n * n; i++) { hA[i] = 1.0f; hB[i] = 1.0f; }   // C should be all n

    float *dA, *dB, *dC;
    cudaMalloc(&dA, bytes); cudaMalloc(&dB, bytes); cudaMalloc(&dC, bytes);
    cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice);

    dim3 threads(TILE, TILE), blocks(n / TILE, n / TILE);
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    float ms_naive, ms_tiled;

    matmul_naive<<<blocks, threads>>>(dA, dB, dC, n); cudaDeviceSynchronize();   // warm up
    cudaEventRecord(s);
    matmul_naive<<<blocks, threads>>>(dA, dB, dC, n);
    cudaEventRecord(e); cudaEventSynchronize(e); cudaEventElapsedTime(&ms_naive, s, e);

    matmul_tiled<<<blocks, threads>>>(dA, dB, dC, n); cudaDeviceSynchronize();   // warm up
    cudaEventRecord(s);
    matmul_tiled<<<blocks, threads>>>(dA, dB, dC, n);
    cudaEventRecord(e); cudaEventSynchronize(e); cudaEventElapsedTime(&ms_tiled, s, e);

    cudaMemcpy(hC, dC, bytes, cudaMemcpyDeviceToHost);

    double flop = 2.0 * n * n * n;
    printf("naive:   %6.3f ms   %7.1f GFLOP/s\n", ms_naive, flop / (ms_naive / 1e3) / 1e9);
    printf("tiled:   %6.3f ms   %7.1f GFLOP/s\n", ms_tiled, flop / (ms_tiled / 1e3) / 1e9);
    printf("speedup: %.2fx\n", ms_naive / ms_tiled);
    printf("check:   C[0] = %.0f  (expected %d)\n", hC[0], n);

    cudaFree(dA); cudaFree(dB); cudaFree(dC); free(hA); free(hB); free(hC);
    return 0;
}

In [ ]:
!nvcc -arch=native -O2 matmul.cu -o matmul && ./matmul

## What you should see

- Both kernels produce `C[0] = 1024` (correct).
- **Tiled is meaningfully faster** — commonly ~2–4× on this size. Same math, same launch config; the only difference is the tiled version reuses data from shared memory instead of hammering main memory.
- Neither is near the GPU's peak — that's what **cuBLAS** (and Tensor Cores) are for. Try it as a bonus: `-lcublas` and call `cublasSgemm`. The lesson isn't "beat cuBLAS," it's *feel why tiling wins*.

## Reflection (write your answers)

1. What speedup did **tiled ÷ naïve** give on your 5090? Why is it not infinite?
2. The naïve kernel re-reads the same row/column many times. Roughly how many times is each value of A re-read, for a 1024×1024 matmul with 16×16 tiles?
3. What are the two `__syncthreads()` calls for? What breaks if you remove them?
4. Change `TILE` to 8 and to 32. What happens to the speed — and why? (Hint: shared-memory size and occupancy.)

### Cleanup

In [ ]:
!rm -f vadd vadd_stride matmul vadd.cu vadd_stride.cu matmul.cu
!echo cleaned